##### VVIDRA Desirability Functions
The main purpose here is to turn "uncomparable" variables comparable by mapping them to a common desirabiltiy scale from 0 - 1.
 
This is accomplished by calculating the deviation from an ideal range. Any time the values are within that range, it is considered to have perfect desirability, and any deviation is translated to a penalty function. The purpose of this notebook to to determine the terms of this initial-order tranformation such that it may be further developed and integrated into the VVIDRA digital twin architecture.  

*More explicit:* The desirability model maps heterogeneous markers (different units, ranges, and clinical meaning) onto a common, unitless scale **d** [0, 1], where 1==fully desirable and 0==fully undesirable. Comparability dos not come from normalization alone, but from shared anchor semantics: d==1 always means "at target", d≈≈0.37 always means "at the limit of tolerance", and d→0 always means maximally unacceptable. The model is a static evaluation of a state. Response dynamics and reachability belong to the downstream feasibility/predictive layers.

##### Transformation function:
*Function that normalizes values such that equal distance means equal physiological significance*

In [1]:
import math

def transform_identity(x):
    tx = x
    return tx

def transform_log(x):
    tx = math.log10(x)
    return tx


##### Deviation function:
*For an ideal rangle [a,b], deviation d=max(0,t(a)-t(x), t(x)-t(b)); where x==raw marker value and t==transformation scaling function from before*

In [2]:
def deviation(x, a, b, transform):
    if transform == "identity":
        tx = transform_identity(x)
        ta = transform_identity(a)
        tb = transform_identity(b)
    elif transform == "log":
        tx = transform_log(x)
        ta = transform_log(a)
        tb = transform_log(b)
    else:
        raise ValueError("ERROR: Marker response curve shape not valid!")

    d = max(0, ta - tx, tx - tb)
    return d

In [3]:
deviation(100, 5, 10, "log")

1.0

##### Desirability function:
*Takes the deviation and contextualizes it based on specific marker properties and measurement sensitivity*

In [4]:
import math

def desirability(x,a, b, transform, s, p):
    d = deviation(x, a, b, transform)
    desire_score = (math.exp(-(d/s)*p))
    return desire_score

In [5]:
desirability(100, 5, 10, "identity", 1, 2)

6.714184288211594e-79

##### Full Look:  

In [6]:
import math

def transform_identity(x):
    tx = x
    return tx

def transform_log(x):
    tx = math.log(x)
    return tx

def deviation(x, a, b, transform):
    if transform == "identity":
        tx = transform_identity(x)
        ta = transform_identity(a)
        tb = transform_identity(b)
    elif transform == "log":
        tx = transform_log(x)
        ta = transform_log(a)
        tb = transform_log(b)
    else:
        raise ValueError("ERROR: Marker response curve shape not valid!")

    d = max(0, ta - tx, tx - tb)
    return d

def desirability(x,a, b, transform, s, p):
    d = deviation(x, a, b, transform)
    desire_score = math.exp(-(d/s)**p)
    return desire_score


In [7]:
a = 139
b = 141
s = 2
p = 2
transform = "identity"

for x in [140, 142, 137, 143.5, 145]:
    d = desirability(x, a, b, transform, s, p)
    print(f"sodium= {x:>5} mmol/L ----> d = {d:.2f}")

sodium=   140 mmol/L ----> d = 1.00
sodium=   142 mmol/L ----> d = 0.78
sodium=   137 mmol/L ----> d = 0.37
sodium= 143.5 mmol/L ----> d = 0.21
sodium=   145 mmol/L ----> d = 0.02
